# ARTICLE2_NB2: Model faults F1-F6

**This notebook produces results for the fault-injection study (see README).**

Protocol: *study protocol* (frozen 11 Nov 2026), Section 3 (faults) and Section 4 (outcomes and decision rules).

**What it does**
- **CELL 2 - ARFIMA checks.** Re-estimates the v2 ARFIMA and checks that it reproduces the snapshot forecasts. Then builds the ARFIMA used by F1 (silent estimation failure) and F2 (static 501-step forecast).
- **CELL 3 - Same-environment reference.** Retrains the five seeds (S0-S4) and checks that they match the NB1 Part C files. Faults are only ever compared with references from the same environment.
- **CELL 4 - Fault runs.** Injects each fault **alone**, for every seed (6 faults x 5 seeds = 30 runs). The run resumes where it stopped if Colab disconnects.
- **CELL 5 - Outcomes.** Delta MAE ratio (primary), apparent improvement, significance counts, Kendall's tau, materiality flags and the mechanism checks.

**Fault code:** `03_Code/article2_faults.py`. It imports the reference code (`article2_pipeline.py`) and never edits it.
**Inputs and outputs:** reads only `Article 2/02_Inputs_v2_snapshot` and `Article 2/05_Outputs/reference`; writes only `Article 2/05_Outputs/faults`.
**Runtime:** about 20-30 minutes on a T4 GPU. Run NB1 Part C first.

## CELL 0: Environment setup (Article 2 folder only)

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

%cd "/content/drive/MyDrive/Article 2"

import os, sys
from pathlib import Path
ROOT = Path(os.getcwd())
assert ROOT.name == "Article 2", f"Wrong folder: {ROOT}"
CODE_DIR = ROOT / "03_Code"
for f in ["article2_pipeline.py", "article2_faults.py"]:
    assert (CODE_DIR / f).exists(), f"{f} missing from {CODE_DIR}"
sys.path.insert(0, str(CODE_DIR))
print("Working directory:", ROOT)

## CELL 1: Imports, versions and settings

In [ ]:
import json, time, pickle, importlib, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
import xgboost as xgb
import sklearn, scipy, statsmodels
warnings.filterwarnings("ignore")

import article2_pipeline as ap
import article2_faults as af
importlib.reload(ap); importlib.reload(af)

P = ap.paths(ROOT)
OUT = P["OUT"]
REF_DIR = OUT / "reference"
FDIR = OUT / "faults"
CACHE = FDIR / "_reference_cache"
for d in [FDIR, CACHE, OUT / "logs"]:
    d.mkdir(parents=True, exist_ok=True)

SEEDS = {"S0": 42, "S1": 101, "S2": 202, "S3": 303, "S4": 404}   # Protocol Section 2 (frozen)
assert (REF_DIR / "reference_seed_range.csv").exists(), "Run ARTICLE2_NB1 Part C first (reference seed range missing)."

VERSIONS = {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
            "tensorflow": tf.__version__, "xgboost": xgb.__version__, "scikit-learn": sklearn.__version__,
            "scipy": scipy.__version__, "statsmodels": statsmodels.__version__,
            "gpu": [d.name for d in tf.config.list_physical_devices("GPU")],
            "run_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
for k, v in VERSIONS.items():
    print(f"{k:14s}: {v}")
print("Seeds:", SEEDS)
print("Input snapshot:", ap.check_input_snapshot(P))

data = ap.load_data(P)
arfima_snap = ap.load_arfima(P)

## CELL 2: ARFIMA checks and the ARFIMA used by F1 and F2 (about 2-3 minutes, CPU)

1. **Reference refit:** the v2 grid search (54 candidates per stock, real fractional differencing) must reproduce the snapshot ARFIMA forecasts.
2. **F1:** the same grid through the v1 route (`statsmodels` ARIMA with d inside `order`). Every non-integer d raises an error, which a `try/except` swallows without logging. The column *Fits_dropped_silently* is our instrument; a researcher running v1 would never see it.
3. **F2:** one static forecast from the last training day. Validation gets steps 1-503; test gets steps 1-501 placed on the test dates.

In [ ]:
t0 = time.time()
ar_ref, ar_models, ar_ref_log = af.fit_arfima(data)
ar_f1, _, ar_f1_log = af.fit_arfima(data, fault="F1")
ar_f2 = af.static_arfima_tables(data, ar_models)
print(f"ARFIMA estimation: {time.time() - t0:.0f} s")

refit_check = af.compare_arfima(ar_ref, arfima_snap)
ARFIMA_REFIT_MAX_DIFF = float(refit_check["Max_abs_diff_USD"].max())
rows_ok = bool((refit_check.Rows_a == refit_check.Rows_b).all())
VERDICT_ARFIMA = "EXACT" if (ARFIMA_REFIT_MAX_DIFF < 1e-6 and rows_ok) else "DRIFT"
print()
print("Reference refit vs snapshot (USD):")
print(refit_check.to_string(index=False))
print("ARFIMA refit verdict:", VERDICT_ARFIMA)

selection = pd.concat([ar_ref_log, ar_f1_log], ignore_index=True)
print()
print("Selected models (REF = v2 route, F1 = v1 route with silent failures):")
print(selection.round(2).to_string(index=False))

(FDIR / "F1").mkdir(exist_ok=True); (FDIR / "F2").mkdir(exist_ok=True)
refit_check.to_csv(FDIR / "arfima_refit_check.csv", index=False)
selection.to_csv(FDIR / "F1" / "F1_arfima_selection.csv", index=False)
af.compare_arfima(ar_f2, ar_ref).to_csv(FDIR / "F2" / "F2_arfima_vs_reference.csv", index=False)
for t in ap.TICKERS:
    ar_f1[t].to_csv(FDIR / "F1" / f"{t}_F1_arfima_predictions.csv")
    ar_f2[t].to_csv(FDIR / "F2" / f"{t}_F2_arfima_predictions.csv")

# Which ARFIMA is the reference for F1 and F2?
# EXACT: the snapshot (as in NB1). DRIFT: the in-session refit, so that F1/F2 differ from their
# reference only by the fault (an extra REF_ARFIMA_REFIT run is then made in CELL 3).
ARFIMA_FOR_REF = arfima_snap if VERDICT_ARFIMA == "EXACT" else ar_ref
print()
print("Reference ARFIMA for F1/F2:", "snapshot" if VERDICT_ARFIMA == "EXACT" else "in-session refit (record in deviation log)")

## CELL 3: Same-environment reference, five seeds (about 8-10 minutes on a T4)

Retrains LSTM and XGBoost (base and serial) for each seed, rebuilds every architecture and compares with the NB1 Part C files. The trained parts are cached in `05_Outputs/faults/_reference_cache` so a disconnect does not repeat this step.

In [ ]:
REF, ref_check, ref_metrics, ref_dm = {}, [], [], []
for label, seed in SEEDS.items():
    t0 = time.time()
    cfile = CACHE / f"seed_{seed}.pkl"
    if cfile.exists():
        with open(cfile, "rb") as f:
            base, serial = pickle.load(f)
        how = "cache"
    else:
        base, _ = ap.train_base(data, seed)
        serial, _ = ap.run_serial(data, arfima_snap, seed)
        with open(cfile, "wb") as f:
            pickle.dump((base, serial), f)
        how = "trained"
    REF[seed] = {"base": base, "serial": serial, "arfima": arfima_snap}
    comb, _, _ = af.assemble(base, arfima_snap, serial, seed)
    saved = {t: pd.read_csv(REF_DIR / f"seed_{seed}" / f"{t}_test_all_models.csv", parse_dates=["Date"], index_col="Date")
             for t in ap.TICKERS}
    diff = af.max_table_diff(comb, saved)
    ref_check.append({"Seed": seed, "Source": how, "Max_abs_diff_vs_NB1_USD": diff, "Match": diff < 1e-6})
    REF[seed]["combined"] = comb
    ref_metrics.append(af.metrics_long(comb, "REF", seed))
    ref_dm.append(af.dm_sets(comb).assign(Run="REF", Seed=seed))
    if VERDICT_ARFIMA != "EXACT":
        s_refit, _ = ap.run_serial(data, ar_ref, seed)
        REF[seed]["serial_refit"] = s_refit
        REF[seed]["combined_refit"], _, _ = af.assemble(base, ar_ref, s_refit, seed)
    print(f"{label} (seed {seed}) {how} in {time.time() - t0:.0f} s | max diff vs NB1: {diff:.2e}")

ref_check = pd.DataFrame(ref_check)
ref_metrics = pd.concat(ref_metrics, ignore_index=True)
ref_dm = pd.concat(ref_dm, ignore_index=True)
ref_check.to_csv(FDIR / "reference_same_environment_check.csv", index=False)
ref_metrics.to_csv(FDIR / "reference_metrics_long.csv", index=False)
ref_dm.to_csv(FDIR / "reference_dm_sets.csv", index=False)
SAME_ENV = bool(ref_check["Match"].all())
print()
print(ref_check.to_string(index=False))
print("Same environment as NB1 Part C:", SAME_ENV)
if not SAME_ENV:
    print("Not identical to NB1. Faults are compared with the references re-run in THIS session (still same-seed, same-environment). Record in the deviation log.")

# The F4 code path must reproduce the reference XGBoost exactly when the leak is switched off
xb, _ = af.xgb_base(data, REF[42]["base"], 42, leak=False)
F4_PATH_DIFF = max(float(np.max(np.abs(xb[t]["XGB"] - REF[42]["base"][t]["XGB"]))) for t in ap.TICKERS)
print(f"F4 code path with leak off vs reference XGBoost (seed 42): max diff {F4_PATH_DIFF:.2e}")
assert F4_PATH_DIFF < 1e-9, "F4 code path does not reproduce the reference. Stop and report."

## CELL 4: Fault runs - each fault alone, five seeds (about 10-15 minutes on a T4)

Only the parts a fault touches are retrained (F1, F2: Serial; F4: standalone XGBoost). Everything else is the same-seed reference from CELL 3. A finished run is skipped on restart.
**No repairs:** if a fault breaks a run, the error is recorded as a result (Protocol Section 4).

In [ ]:
scalers = af.minmax_scalers(data)
FAULT_ARFIMA = {"F1": ar_f1, "F2": ar_f2}
status = []
for fault in af.MODEL_FAULTS:
    for label, seed in SEEDS.items():
        rdir = FDIR / fault / f"seed_{seed}"
        if (rdir / "done.json").exists():
            status.append({"Fault": fault, "Seed": seed, "Status": "done earlier"})
            continue
        rdir.mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        try:
            comb, logs = af.run_fault(fault, seed, data, REF[seed], fault_arfima=FAULT_ARFIMA.get(fault), scalers=scalers)
        except Exception as e:
            with open(rdir / "error.json", "w") as f:
                json.dump({"fault": fault, "seed": seed, "error": repr(e)}, f, indent=2)
            status.append({"Fault": fault, "Seed": seed, "Status": f"ERROR: {repr(e)[:80]}"})
            print(f"{fault} seed {seed}: ERROR recorded ({repr(e)[:80]})")
            continue
        for t in ap.TICKERS:
            comb[t].to_csv(rdir / f"{t}_test_all_models.csv")
        af.metrics_long(comb, fault, seed).to_csv(rdir / "metrics_long.csv", index=False)
        dm = af.dm_sets(comb).assign(Run=fault, Seed=seed)
        dm.to_csv(rdir / "dm_sets.csv", index=False)
        for k, v in logs.items():
            v.assign(Fault=fault).to_csv(rdir / f"log_{k}.csv", index=False)
        with open(rdir / "done.json", "w") as f:
            json.dump({"fault": fault, "seed": seed, "seconds": round(time.time() - t0, 1),
                       "finished": datetime.now().isoformat(timespec="seconds")}, f)
        status.append({"Fault": fault, "Seed": seed, "Status": "ok"})
        print(f"{fault} seed {seed}: {time.time() - t0:.0f} s | significant (Holm): {af.sig_counts(dm)}")
status = pd.DataFrame(status)
print()
print(status.pivot(index="Fault", columns="Seed", values="Status").to_string())

## CELL 5: Outcomes (Protocol Section 4)

- **Delta MAE ratio** (primary) = fault minus same-seed reference. Mean and min-max over five seeds. Never the best seed alone.
- **Material fault** if (a) the mean MAE ratio over seeds falls outside the reference seed range for any model (stock mean), or (b) the significance count changes, or (c) Kendall's tau < 1.
- **Apparent improvement** = the percentage gain over ARFIMA (and over persistence) a researcher would report under the fault, minus the same gain under the reference.

In [ ]:
def load_runs(name):
    frames = []
    for fault in af.MODEL_FAULTS:
        for seed in SEEDS.values():
            f = FDIR / fault / f"seed_{seed}" / name
            if f.exists():
                frames.append(pd.read_csv(f))
    return pd.concat(frames, ignore_index=True)

fm = load_runs("metrics_long.csv")
fdm = load_runs("dm_sets.csv")
ref_m = ref_metrics.copy()
if VERDICT_ARFIMA != "EXACT":   # F1/F2 are judged against the in-session ARFIMA refit
    ref_m_refit = pd.concat([af.metrics_long(REF[s]["combined_refit"], "REF_ARFIMA_REFIT", s) for s in SEEDS.values()])

def ref_for(fault):
    return ref_m_refit if (fault in ("F1", "F2") and VERDICT_ARFIMA != "EXACT") else ref_m

# Primary outcome: Delta MAE ratio per model, stock and seed (plus the stock mean)
rows = []
for fault in af.MODEL_FAULTS:
    a = fm[fm.Run == fault]
    b = ref_for(fault)
    m = a.merge(b[["Seed", "Ticker", "Model", "MAE_ratio"]], on=["Seed", "Ticker", "Model"], suffixes=("", "_ref"))
    m["Delta"] = m["MAE_ratio"] - m["MAE_ratio_ref"]
    mean_stock = m.groupby(["Seed", "Model"])[["MAE_ratio", "MAE_ratio_ref", "Delta"]].mean().reset_index().assign(Ticker="Mean")
    rows.append(pd.concat([m, mean_stock]).assign(Fault=fault))
delta = pd.concat(rows, ignore_index=True)
delta = delta[delta.Model != "Persistence"]

summary = (delta.groupby(["Fault", "Model", "Ticker"])
           .agg(Fault_ratio=("MAE_ratio", "mean"), Delta_mean=("Delta", "mean"),
                Delta_min=("Delta", "min"), Delta_max=("Delta", "max")).reset_index())
seed_range = pd.read_csv(REF_DIR / "reference_seed_range.csv").rename(columns={"Stock": "Ticker"})
summary = summary.merge(seed_range[["Model", "Ticker", "min", "max"]].rename(columns={"min": "Ref_min", "max": "Ref_max"}),
                        on=["Model", "Ticker"], how="left")
# Compared at three decimals (protocol rounding rule): ARFIMA has a zero-width seed range, so a raw
# comparison would flag differences far below the reported precision.
summary["Outside_ref_range"] = ((summary.Fault_ratio.round(3) < summary.Ref_min.round(3))
                                | (summary.Fault_ratio.round(3) > summary.Ref_max.round(3)))
summary.to_csv(FDIR / "faults_delta_summary.csv", index=False)

# Significance counts and ranking agreement, per fault and seed
sig_rows = []
for fault in af.MODEL_FAULTS:
    for seed in SEEDS.values():
        d = fdm[(fdm.Run == fault) & (fdm.Seed == seed)]
        if d.empty:
            continue
        r = ref_dm[ref_dm.Seed == seed]
        fc, rc = af.sig_counts(d), af.sig_counts(r)
        tau = af.ranking_tau(fm[(fm.Run == fault) & (fm.Seed == seed)], ref_for(fault)[ref_for(fault).Seed == seed])
        sig_rows.append({"Fault": fault, "Seed": seed, "Tau": tau,
                         **{f"Sig_{k}": fc.get(k, 0) for k in ["primary", "vs_persistence", "vs_ARFIMA"]},
                         **{f"Ref_{k}": rc.get(k, 0) for k in ["primary", "vs_persistence", "vs_ARFIMA"]}})
sig = pd.DataFrame(sig_rows)
sig["Count_changed"] = ((sig.Sig_primary != sig.Ref_primary) | (sig.Sig_vs_persistence != sig.Ref_vs_persistence)
                        | (sig.Sig_vs_ARFIMA != sig.Ref_vs_ARFIMA))
sig.to_csv(FDIR / "faults_significance_and_tau.csv", index=False)
# Context only (not a decision rule): how far seed noise alone moves the ranking, S1-S4 vs S0
REF_TAU_NOISE = {s: af.ranking_tau(ref_m[ref_m.Seed == s], ref_m[ref_m.Seed == 42]) for s in list(SEEDS.values())[1:]}

# Materiality (decision rules fixed in the protocol)
mat = []
for fault in af.MODEL_FAULTS:
    s = summary[(summary.Fault == fault) & (summary.Ticker == "Mean")]
    g = sig[sig.Fault == fault]
    out_models = s.loc[s.Outside_ref_range, "Model"].tolist()
    mat.append({"Fault": fault, "Name": af.FAULT_NAMES[fault],
                "Models_outside_ref_range": len(out_models), "Which": ", ".join(out_models),
                "Seeds_sig_count_changed": int(g.Count_changed.sum()), "Tau_min": round(float(g.Tau.min()), 3),
                "Seeds_tau_below_1": int((g.Tau < 1 - 1e-12).sum()),
                "Material": bool(out_models) or bool(g.Count_changed.any()) or bool((g.Tau < 1 - 1e-12).any())})
mat = pd.DataFrame(mat)
mat.to_csv(FDIR / "faults_materiality.csv", index=False)

# Apparent improvement over ARFIMA and over persistence (architectures), fault minus reference
def gains(mdf):
    p = mdf.pivot_table(index=["Run", "Seed", "Ticker"], columns="Model", values="MAE").reset_index()
    out = []
    for a in af.ARCHITECTURES:
        out.append(pd.DataFrame({"Run": p.Run, "Seed": p.Seed, "Ticker": p.Ticker, "Architecture": a,
                                 "Gain_vs_ARFIMA_pct": 100 * (p["ARFIMA"] - p[a]) / p["ARFIMA"],
                                 "Gain_vs_Persistence_pct": 100 * (p["Persistence"] - p[a]) / p["Persistence"]}))
    return pd.concat(out, ignore_index=True)

app = []
for fault in af.MODEL_FAULTS:
    gf, gr = gains(fm[fm.Run == fault]), gains(ref_for(fault))
    m = gf.merge(gr.drop(columns="Run"), on=["Seed", "Ticker", "Architecture"], suffixes=("", "_ref"))
    m["Apparent_vs_ARFIMA_pp"] = m.Gain_vs_ARFIMA_pct - m.Gain_vs_ARFIMA_pct_ref
    m["Apparent_vs_Persistence_pp"] = m.Gain_vs_Persistence_pct - m.Gain_vs_Persistence_pct_ref
    app.append(m.assign(Fault=fault))
app = pd.concat(app, ignore_index=True)
app.to_csv(FDIR / "faults_apparent_improvement_long.csv", index=False)
app_sum = app.groupby(["Fault", "Architecture"])[["Gain_vs_ARFIMA_pct", "Gain_vs_ARFIMA_pct_ref", "Apparent_vs_ARFIMA_pp",
                                                    "Apparent_vs_Persistence_pp"]].mean().reset_index()
app_sum.to_csv(FDIR / "faults_apparent_improvement_summary.csv", index=False)

pd.set_option("display.width", 200)
print("MATERIALITY (Protocol Section 4 decision rules)")
print(mat.to_string(index=False))
print()
print("DELTA MAE RATIO, stock mean: mean over seeds [min, max]")
tab = summary[summary.Ticker == "Mean"].copy()
tab["Delta"] = tab.apply(lambda r: f"{r.Delta_mean:+.3f} [{r.Delta_min:+.3f}, {r.Delta_max:+.3f}]", axis=1)
print(tab.pivot(index="Model", columns="Fault", values="Delta").loc[af.ALL_MODELS].to_string())
print()
print("Seed noise alone, Kendall's tau of reference seeds S1-S4 vs S0:", {k: round(v, 3) for k, v in REF_TAU_NOISE.items()})
print("SIGNIFICANCE COUNTS (Holm) and Kendall's tau, mean over seeds (reference in brackets)")
g = sig.groupby("Fault").mean(numeric_only=True)
for fault, r in g.iterrows():
    print(f"{fault}: primary {r.Sig_primary:.1f} ({r.Ref_primary:.1f}) | vs persistence {r.Sig_vs_persistence:.1f} "
          f"({r.Ref_vs_persistence:.1f}) | vs ARFIMA {r.Sig_vs_ARFIMA:.1f} ({r.Ref_vs_ARFIMA:.1f}) | tau {r.Tau:.2f}")
print()
print("APPARENT IMPROVEMENT over ARFIMA, percentage points (fault minus reference), mean over seeds and stocks")
print(app_sum.pivot(index="Architecture", columns="Fault", values="Apparent_vs_ARFIMA_pp").round(2).to_string())

## CELL 6: Mechanism checks (Protocol Section 4, fault-specific outcomes)

In [ ]:
def load_log(fault, name):
    frames = [pd.read_csv(FDIR / fault / f"seed_{s}" / f"log_{name}.csv") for s in SEEDS.values()
              if (FDIR / fault / f"seed_{s}" / f"log_{name}.csv").exists()]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

print("F1 - selected ARFIMA (no seed dependence):")
print(selection[["Ticker", "Fault", "p", "d", "q", "AIC", "Fits_logged", "Fits_dropped_silently"]].round(2).to_string(index=False))

print()
print("F2 - ARFIMA test MAE ratio, reference vs static forecast (seed independent):")
f2r = summary[(summary.Fault == "F2") & (summary.Model == "ARFIMA")][["Ticker", "Fault_ratio", "Delta_mean"]]
print(f2r.round(3).to_string(index=False))

print()
print("F3 - units on the test set (seed S0):")
print(load_log("F3", "units").query("Seed == 42").round(3).to_string(index=False))

print()
print("F4 - leaked XGBoost: trees kept and R2 (mean over seeds) vs reference:")
f4 = load_log("F4", "xgb").groupby("Ticker")[["XGB_trees_kept", "XGB_train_R2_returns", "XGB_test_R2_returns", "Leak_feature_importance"]].mean()
rb = pd.read_csv(REF_DIR / "reference_base_logs_by_seed.csv").groupby("Ticker")[["XGB_best_iteration", "XGB_train_R2_returns"]].mean()
f4["Ref_trees_kept"] = rb["XGB_best_iteration"] + 1
f4["Ref_train_R2_returns"] = rb["XGB_train_R2_returns"]
print(f4.round(3).to_string())

for fault in ["F4", "F5"]:
    print()
    print(f"{fault} - Parallel weights and Stacking coefficients (mean over seeds):")
    w = load_log(fault, "weights").groupby(["Ticker", "Scheme"])[["w_ARFIMA", "w_LSTM", "w_XGB"]].mean()
    print(w.round(3).to_string())
    print(load_log(fault, "stacking").groupby("Ticker")[["b0", "b_ARFIMA", "b_LSTM", "b_XGB"]].mean().round(3).to_string())

print()
print("F6 - Serial versus standalone models (all seeds):")
eq = load_log("F6", "equality")
print(eq.groupby("Ticker")[["Max_abs_diff_Final_vs_XGBoost", "Corr_Final_vs_XGBoost", "Max_abs_diff_Stage2_vs_LSTM"]].max().to_string())

## CELL 7: CHECKPOINT and run log

In [ ]:
done = sum((FDIR / f / f"seed_{s}" / "done.json").exists() for f in af.MODEL_FAULTS for s in SEEDS.values())
errors = sum((FDIR / f / f"seed_{s}" / "error.json").exists() for f in af.MODEL_FAULTS for s in SEEDS.values())
checks = {
    "arfima_refit_reproduces_snapshot": VERDICT_ARFIMA == "EXACT",
    "reference_same_environment_as_NB1": SAME_ENV,
    "f4_code_path_exact_with_leak_off": F4_PATH_DIFF < 1e-9,
    "all_30_runs_finished_or_error_recorded": done + errors == 30,
    "f1_dropped_all_fractional_candidates": bool((ar_f1_log.Fits_dropped_silently >= 45).all()),
    "f1_selected_d_zero": bool((ar_f1_log.d == 0).all()),
    "f6_serial_equals_xgboost": bool(load_log("F6", "equality").Max_abs_diff_Final_vs_XGBoost.max() < 1e-6),
}
print(pd.Series(checks).to_string())
print(f"Runs finished: {done} | errors recorded as results: {errors}")

log = {"notebook": "ARTICLE2_NB2_Model_Faults", "purpose": "Article 2 only", "root": str(ROOT),
       "run_at": datetime.now().isoformat(timespec="seconds"), "versions": VERSIONS, "seeds": SEEDS,
       "arfima_refit_verdict": VERDICT_ARFIMA, "arfima_refit_max_diff_usd": ARFIMA_REFIT_MAX_DIFF,
       "same_environment_as_nb1": SAME_ENV, "runs_done": done, "runs_error": errors,
       "material_faults": mat.loc[mat.Material, "Fault"].tolist(), "ref_tau_noise": REF_TAU_NOISE, "checks": checks}
with open(OUT / "logs" / "ARTICLE2_NB2_run_log.json", "w") as f:
    json.dump(log, f, indent=2, default=str)

soft = [k for k in ["arfima_refit_reproduces_snapshot", "reference_same_environment_as_NB1"] if not checks[k]]
hard = [k for k, v in checks.items() if not v and k not in soft]
if soft:
    print("Needs a deviation-log entry (not a stop):", soft)
assert not hard, f"CHECKPOINT FAILED: {hard}"
print()
print("Hard checks: ALL PASSED")
print("Please share the outputs of CELLS 2, 3, 5, 6 and 7.")